# 2. Exploratory data analysis

**Question:** What patterns and anomalies are present?

We examine diagnosis composition, expression distributions, correlated targets,
and assay batches. These are retrospective descriptive analyses added after the
presentation. The original technical-validation results have already been seen.
Nothing here changes a fitted model, a threshold, or specimen inclusion.

Training specimens determine variable-target selection and the PCA transformation.
Screening and technical-validation specimens are projected using those same
fitted parameters. Views across cohorts describe differences rather than testing
a new model.

## Setup

Run with this project's Python 3.12 environment (uv sync --frozen --group notebooks).
The [notebook guide](README.md) has launch and clean-checkout instructions.
Each notebook runs independently. Figures and aggregate tables go to a new
directory under results/notebooks/; complete specimen tables stay in memory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src/kidney_biopsy").is_dir():
    raise RuntimeError("Start Jupyter from this project root or its notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from kidney_biopsy import map_diagnoses, normalize_counts, read_geo_matrix, read_rcc_archive
from notebooks.notebook_support import (
    BK_TARGETS,
    COLORS,
    DIAGNOSIS_ORDER,
    SHORT_DIAGNOSES,
    SPLIT_ORDER,
    NotebookSession,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4.5),
        "figure.dpi": 110,
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "axes.titlepad": 14,
    }
)
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 3)
session = NotebookSession("02_exploratory_data_analysis")
display(session.verify_inputs())

In [ ]:
raw_dir = ROOT / "data/raw/rejection_public"
deposited_expression, metadata = read_geo_matrix(raw_dir / "GSE212160_series_matrix.txt.gz")
counts, batches = read_rcc_archive(raw_dir / "GSE212160_RAW.tar", specimen_ids=metadata.index)
diagnosis = metadata.histology_diagnosis
y = map_diagnoses(diagnosis)

In [ ]:
X = normalize_counts(counts)
split = session.load_split(metadata)
train_ids = split.index[split.split.eq("train")]
screen_ids = split.index[split.split.eq("discovery_screen")]
validation_ids = split.index[split.split.eq("author_validation")]
X_train = X.loc[train_ids]

## 1. Who is represented in each split?

The saved specimen split preserves the authors' discovery and technical-validation
cohorts. Discovery was split by the four diagnoses with seed 20260915.
The tables below count specimens, not unique patients.

In [ ]:
composition = pd.crosstab(split.split, diagnosis).reindex(
    index=SPLIT_ORDER, columns=DIAGNOSIS_ORDER
)
prevalence = pd.DataFrame(
    {
        "specimens": split.groupby("split").size(),
        "rejection_cases": y.groupby(split.split).sum(),
        "rejection_fraction": y.groupby(split.split).mean(),
    }
).reindex(SPLIT_ORDER)
display(composition)
display(prevalence)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
composition.set_axis(SHORT_DIAGNOSES, axis=1).plot.bar(
    stacked=True, ax=axes[0], color=COLORS, rot=0
)
axes[0].set(xlabel="", ylabel="Specimens", title="Recorded diagnoses")
axes[0].legend(fontsize=8)
prevalence.rejection_fraction.plot.bar(ax=axes[1], color=COLORS[:3], rot=0)
axes[1].set(
    xlabel="",
    ylabel="Fraction with recorded rejection",
    ylim=(0, 1),
    title="Prevalence differs by cohort",
)
for ax in axes:
    ax.set_xticklabels(["Training", "Screening", "Technical validation"], fontsize=9)
session.figure("diagnosis_composition", fig)
plt.show()
session.table("diagnosis_counts", composition)
session.table("split_prevalence", prevalence)
display(
    Markdown(
        f"**Observed:** rejection prevalence is {y.loc[train_ids].mean():.1%} in training "
        f"and {y.loc[validation_ids].mean():.1%} in technical validation. "
        "This difference matters when interpreting precision and score reliability."
    )
)

## 2. Do identical molecular profiles cross splits?

Distinct specimen IDs alone do not rule out repeated records. We also check exact
normalized profiles. A clean result cannot establish patient or center independence
because those identifiers are unavailable.

In [ ]:
duplicate_profiles = X.duplicated(keep=False)
profile_hash = pd.util.hash_pandas_object(X, index=False)
split_counts = (
    pd.DataFrame({"profile": profile_hash, "split": split.split}).groupby("profile").split.nunique()
)
cross_split_profiles = int(split_counts.gt(1).sum())
assert cross_split_profiles == 0, (
    "Identical profiles cross splits; investigate before interpreting results."
)
print(f"Specimens in duplicate profiles: {int(duplicate_profiles.sum())}")
print(f"Identical profiles spanning multiple splits: {cross_split_profiles}")

## 3. Which targets are sparse or vary little?

The ranking below uses training specimens only. Standard deviation describes
variation on the normalized scale. No target is removed by this notebook.

In [ ]:
target_summary = pd.DataFrame(
    {
        "normalized_mean": X_train.mean(),
        "normalized_sd": X_train.std(),
        "normalized_iqr": X_train.quantile(0.75) - X_train.quantile(0.25),
        "raw_zero_fraction": counts.loc[train_ids, X.columns].eq(0).mean(),
        "distinct_normalized_values": X_train.nunique(),
    }
)
display(target_summary.nsmallest(10, "normalized_sd"))
display(target_summary.nlargest(10, "raw_zero_fraction"))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].hist(target_summary.normalized_sd, bins=35, color=COLORS[0])
axes[0].set(
    xlabel="Normalized standard deviation", ylabel="Assay targets", title="Variation in training"
)
axes[1].hist(target_summary.raw_zero_fraction, bins=30, color=COLORS[1])
axes[1].set(xlabel="Raw zero fraction", ylabel="Assay targets", title="Sparsity in training")
session.figure("target_variation", fig)
plt.show()
session.table("training_target_summary", target_summary)

## 4. How do selected RNA distributions differ across diagnoses?

IFNG is the prespecified single-target benchmark. GBP4 is included because it
led the saved model ranking; CXCL9 and CXCL11 are illustrative companion targets.
This choice is retrospective and is not an independent marker-discovery result.
All boxes below use discovery specimens. The plot shows overlap as well as differences.

In [ ]:
discovery_ids = split.index[split.split.ne("author_validation")]
illustrative_targets = ["IFNG", "GBP4", "CXCL9", "CXCL11"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
for ax, target in zip(axes.flat, illustrative_targets):
    groups = [
        X.loc[discovery_ids[diagnosis.loc[discovery_ids].eq(label)], target]
        for label in DIAGNOSIS_ORDER
    ]
    ax.boxplot(
        groups,
        tick_labels=SHORT_DIAGNOSES,
        showfliers=True,
        flierprops={"markersize": 2, "alpha": 0.3},
    )
    ax.set(title=target, ylabel="Normalized assay value")
    ax.tick_params(axis="x", rotation=15, labelsize=9)
session.figure("rna_by_diagnosis", fig)
plt.show()

## 5. Which targets move together?

The heatmap uses the 12 most variable training targets, selected without labels.
The adjacent table searches all training-target pairs for large absolute Pearson
correlations. Correlated predictors can share or exchange model importance.

In [ ]:
variable_targets = target_summary.normalized_sd.nlargest(12).index
correlation = X_train.corr()
upper = correlation.where(np.triu(np.ones(correlation.shape, dtype=bool), k=1))
pairs = upper.stack().rename("correlation").reset_index()
pairs.columns = ["target_1", "target_2", "correlation"]
pairs["absolute_correlation"] = pairs.correlation.abs()
top_pairs = pairs.nlargest(10, "absolute_correlation")
display(top_pairs)
fig, ax = plt.subplots(figsize=(9, 8), layout="constrained")
im = ax.imshow(correlation.loc[variable_targets, variable_targets], cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(12), variable_targets, rotation=55, ha="right")
ax.set_yticks(range(12), variable_targets)
ax.set_title("Training correlations: 12 most variable targets")
fig.colorbar(im, ax=ax, label="Pearson correlation", shrink=0.8)
session.figure("training_correlations", fig)
plt.show()
session.table("strongest_training_correlations", top_pairs)

## 6. Do broad expression patterns align with diagnosis or cohort?

PCA summarizes many measurements in a few axes. Scaling and PCA are fitted on
training specimens only. Color is added afterward. Separation on this plot does
not establish a validated classifier, and overlapping points can still differ on
other axes.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=2, svd_solver="full").fit(pca_scaler.transform(X_train))
coordinates = pd.DataFrame(
    pca.transform(pca_scaler.transform(X)), index=X.index, columns=["PC1", "PC2"]
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
for label, short, color in zip(DIAGNOSIS_ORDER, SHORT_DIAGNOSES, COLORS):
    ids = diagnosis.index[diagnosis.eq(label)]
    axes[0].scatter(
        coordinates.loc[ids, "PC1"],
        coordinates.loc[ids, "PC2"],
        s=11,
        alpha=0.4,
        color=color,
        label=short,
    )
for label, short, color in zip(
    SPLIT_ORDER, ["Training", "Screening", "Technical validation"], COLORS
):
    ids = split.index[split.split.eq(label)]
    axes[1].scatter(
        coordinates.loc[ids, "PC1"],
        coordinates.loc[ids, "PC2"],
        s=11,
        alpha=0.4,
        color=color,
        label=short,
    )
for ax in axes:
    ax.set(
        xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} of training variance)",
        ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
    )
    ax.legend(fontsize=8)
axes[0].set_title("Same projection, colored by diagnosis")
axes[1].set_title("Same projection, colored by split")
session.figure("pca_diagnosis_and_split", fig)
plt.show()
print(f"First two PCs explain {pca.explained_variance_ratio_.sum():.1%} of training variance.")

## 7. Could assay batches contribute to cohort differences?

Dates and cartridges are available for description, and are excluded from model
predictors. We count overlap rather than assuming that the two cohorts represent
independent patients or centers. A cohort shift can reflect diagnoses, controls,
laboratory conditions, or several of these together.

In [ ]:
batch_rows = []
for field in ["Date", "CartridgeID", "ScannerID"]:
    development = set(batches.loc[discovery_ids, field])
    evaluation = set(batches.loc[validation_ids, field])
    batch_rows.append(
        {
            "field": field,
            "discovery_distinct": len(development),
            "validation_distinct": len(evaluation),
            "shared_distinct": len(development & evaluation),
            "missing": int(batches[field].isna().sum()),
        }
    )
batch_overlap = pd.DataFrame(batch_rows)
display(batch_overlap)
shift = pd.DataFrame(
    {
        "training_median": X_train.median(),
        "validation_median": X.loc[validation_ids].median(),
    }
)
shift["median_difference"] = shift.validation_median - shift.training_median
largest_shift = shift.loc[shift.median_difference.abs().nlargest(10).index]
display(largest_shift)
batch_sizes = batches.groupby([metadata.cohort, "CartridgeID"]).size()
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
for cohort, label, color in zip(
    ["Discovery cohort sample", "Validation cohort sample"],
    ["Discovery", "Technical validation"],
    COLORS,
):
    ax.hist(
        batch_sizes.loc[cohort],
        bins=np.arange(0.5, batch_sizes.max() + 1.5),
        alpha=0.6,
        label=label,
        color=color,
    )
ax.set(xlabel="Specimens per cartridge", ylabel="Cartridges", title="Batch sizes by author cohort")
ax.legend()
session.figure("cartridge_sizes", fig)
plt.show()
session.table("batch_overlap", batch_overlap)
session.table("largest_cohort_median_differences", largest_shift)
display(
    Markdown(
        "**Interpretation:** where dates or cartridges do not overlap, their effects "
        "cannot be separated from cohort using this comparison. Differences in target "
        "medians are descriptive, not evidence of a particular laboratory cause."
    )
)

## 8. Inspect the BK viral measurements

The B-HOT panel includes viral measurements. These two BK targets are not human
genes. Both were prominent in the saved CatBoost ranking, so their distributions
deserve a direct look.

A normalized value of zero means 'count + 1' equals the specimen's geometric
mean housekeeping 'count + 1'. It is **not an infection cutoff**. The reference
below is descriptive and was chosen after reviewing the model.

In [ ]:
both_high = X[BK_TARGETS].gt(0).all(axis=1)
bk_summary = pd.DataFrame(
    {"cohort": metadata.cohort, "diagnosis": diagnosis, "both_above_zero": both_high}
)
bk_summary = bk_summary.groupby(["cohort", "diagnosis"]).both_above_zero.agg(
    specimens="size", both_above_zero="sum"
)
display(bk_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
for ax, target in zip(axes, BK_TARGETS):
    ax.boxplot(
        [X.loc[diagnosis.eq(label), target] for label in DIAGNOSIS_ORDER],
        tick_labels=SHORT_DIAGNOSES,
        flierprops={"markersize": 3, "alpha": 0.4},
    )
    ax.axhline(0, color=COLORS[1], linestyle="--", linewidth=1)
    ax.set(title=f"{target} (all specimens)", ylabel="Normalized assay value")
    ax.tick_params(axis="x", rotation=15, labelsize=9)
session.figure("bk_targets", fig)
plt.show()
session.table("bk_signal_groups", bk_summary)
display(
    Markdown(
        f"**Observed:** {int(both_high.sum())} specimens have both BK values above zero; "
        f"{int((both_high & y.eq(1)).sum())} have recorded rejection. "
        "This association may reflect the composition of the no-rejection group. "
        "The public metadata do not identify infection status for these specimens."
    )
)

## 9. Findings to carry into model interpretation

- Compare errors on the same rows and report denominators because prevalence differs.
- Inspect correlated targets before interpreting a ranked feature as uniquely important.
- Keep cohort and assay-batch effects in view when describing generalization.
- Distinguish viral measurements from human RNA targets.
- Preserve all specimens here; these plots supply descriptions, not new exclusion rules.

See the [source-study review](../docs/references/STUDY_AUDIT_20260919.md) and
[earlier viral review](../results/analysis/20260915_viral/REPORT.md) for source
evidence and the public metadata's limits.

In [ ]:
session.finish(
    {
        "cross_split_duplicate_profiles": cross_split_profiles,
        "constant_training_targets": int(target_summary.normalized_sd.eq(0).sum()),
        "first_two_pc_variance": float(pca.explained_variance_ratio_.sum()),
        "bk_both_above_zero": int(both_high.sum()),
        "bk_both_above_zero_rejection": int((both_high & y.eq(1)).sum()),
    },
    {
        "pca": "StandardScaler and 2-component full-SVD PCA fitted on training only",
        "correlation_targets": variable_targets.tolist(),
        "excluded_specimens": 0,
    },
)